# 🎙️ Piper TTS Custom Voice Training (Modern OHF-Voice Engine)
Train your own custom Piper TTS voice model on a free Google Colab GPU (T4), then export to `.onnx` for instant offline CPU playback in Paraclea.

### Fixes applied in this version:
- **PyTorch 2.6+ unpickler fix**: Overrides `torch.load` to `weights_only=False` and allowlists `pathlib.PosixPath`.
- **`espeakbridge` fix**: Builds native extension with `scikit-build` and `libespeak-ng-dev`.
- **`_parse_ckpt_path` fix**: Bypasses Lightning's hyperparameter validation for legacy checkpoints.

In [ ]:
# @title 1. Install Dependencies & Compile Monotonic Aligner
import os, sys

print("Installing build tools and system audio packages...")
!apt-get update -qq && apt-get install -y -qq build-essential cmake ninja-build espeak-ng libespeak-ng-dev git-lfs libasound2-dev

%cd /content
if not os.path.exists('/content/piper1-gpl'):
    print("Cloning active OHF-Voice piper1-gpl repository...")
    !git clone https://github.com/OHF-Voice/piper1-gpl.git
else:
    print("Repository already present.")

%cd /content/piper1-gpl
!pip install --upgrade -q pip setuptools wheel scikit-build piper-tts
!pip install -q -e ".[train]"

print("Building espeakbridge C extension...")
!cp -f /usr/local/lib/python*/dist-packages/piper/espeakbridge* /content/piper1-gpl/src/piper/ 2>/dev/null || true
!python3 setup.py build_ext --inplace

print("Compiling monotonic alignment extension...")
!bash build_monotonic_align.sh
%cd /content
print("\n Environment fully configured for Piper voice training!")

In [ ]:
# @title 2. Download Pretrained Medium Checkpoint (845 MB)
import os
!mkdir -p /content/checkpoints
%cd /content/checkpoints

ckpt_file = 'en_US-lessac-medium.ckpt'
if not os.path.exists(ckpt_file) or os.path.getsize(ckpt_file) < 500 * 1024 * 1024:
    print("Downloading base checkpoint from Hugging Face...")
    !wget -c -L -O en_US-lessac-medium.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt"

size_mb = os.path.getsize(ckpt_file) / (1024 * 1024)
print(f"\n Downloaded checkpoint size: {size_mb:.2f} MB")
assert size_mb > 500, "Error: Checkpoint file is incomplete or corrupted!"
%cd /content
print("Base checkpoint verified and ready for fine-tuning!")

In [ ]:
# @title 3. Upload & Prepare Voice Dataset (Diana.wav)
import os, shutil
from google.colab import files

os.makedirs('/content/dataset/wavs', exist_ok=True)
dest_wav_path = '/content/dataset/wavs/Diana.wav'

# Enter the exact transcript spoken in Diana.wav
transcription = "In the quiet stillness of the evening, wisdom speaks not in a shout, but in a gentle whisper." # @param {type:"string"}

if not os.path.exists(dest_wav_path):
    print("Please upload your 'Diana.wav' file:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.lower().endswith('.wav'):
            with open(dest_wav_path, 'wb') as f:
                f.write(uploaded[fn])
            print(f"Saved {fn} as {dest_wav_path}")
            break
else:
    print("Diana.wav already found in /content/dataset/wavs/")

# Write metadata CSV mapping audio to transcription
with open('/content/dataset/metadata.csv', 'w', encoding='utf-8') as f:
    f.write(f"Diana.wav|{transcription}\n")

print("Dataset prepared successfully:")
print(f"- Audio: {dest_wav_path}")
print(f"- Transcription: {transcription}")

In [ ]:
# @title 4. Fine-Tune Piper Model on GPU
import os, glob

%cd /content/piper1-gpl

# 1. Inject PyTorch 2.6+ unpickler patch directly into piper.train entrypoint
main_path = "/content/piper1-gpl/src/piper/train/__main__.py"
with open(main_path, "r") as f:
    main_code = f.read()
patch_header = """import torch, pathlib
try:
    torch.serialization.add_safe_globals([pathlib.PosixPath, pathlib.WindowsPath])
except Exception:
    pass
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _orig_load(*args, **kwargs)
torch.load = _safe_load
"""
if "_safe_load" not in main_code:
    with open(main_path, "w") as f:
        f.write(patch_header + "\n" + main_code)
    print("Patched piper.train entrypoint for PyTorch 2.6+ checkpoint loading!")

# 2. Bypass Lightning _parse_ckpt_path hyperparameter validator
cli_files = glob.glob("/usr/local/lib/python*/dist-packages/lightning/pytorch/cli.py")
for cf in cli_files:
    with open(cf, 'r') as f:
        content = f.read()
    content = content.replace('weights_only=True', 'weights_only=False')
    old_def = "def _parse_ckpt_path(self) -> None:"
    new_def = "def _parse_ckpt_path(self) -> None:\n        return"
    if old_def in content and new_def not in content:
        content = content.replace(old_def, new_def)
    with open(cf, 'w') as f:
        f.write(content)
    print(f"Patched {cf} successfully!")

os.makedirs('/content/training_dir', exist_ok=True)

# 3. Run fine-tuning against base checkpoint
!python3 -m piper.train fit \
  --data.voice_name "diana" \
  --data.csv_path "/content/dataset/metadata.csv" \
  --data.audio_dir "/content/dataset/wavs" \
  --data.espeak_voice "en-us" \
  --data.cache_dir "/content/training_dir/cache" \
  --data.config_path "/content/training_dir/config.json" \
  --ckpt_path "/content/checkpoints/en_US-lessac-medium.ckpt" \
  --trainer.default_root_dir "/content/training_dir" \
  --trainer.max_epochs 500 \
  --trainer.accelerator gpu \
  --trainer.devices 1 \
  --data.batch_size 16

In [ ]:
# @title 5. Export to ONNX for Paraclea CPU Playback
import glob, os
from google.colab import files

%cd /content/piper1-gpl
ckpts = sorted(glob.glob("/content/training_dir/**/checkpoints/*.ckpt", recursive=True)) + \
        sorted(glob.glob("/content/piper1-gpl/**/checkpoints/*.ckpt", recursive=True))
if ckpts:
    latest_ckpt = ckpts[-1]
    print(f"Exporting checkpoint: {latest_ckpt}")
    !python3 -m piper.train.export_onnx \
      --checkpoint "{latest_ckpt}" \
      --output-file "/content/diana.onnx"

    if os.path.exists("/content/diana.onnx"):
        print("Downloading diana.onnx and diana.onnx.json...")
        files.download("/content/diana.onnx")
        if os.path.exists("/content/diana.onnx.json"):
            files.download("/content/diana.onnx.json")
else:
    print("No checkpoint found yet. Run Step 4 first!")